In [0]:
from pyspark.sql.functions import *
stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", 
            "abfss://bronze@travelappprojectstorage.dfs.core.windows.net/_schemas/latest_trilateration")
    .load("abfss://bronze@travelappprojectstorage.dfs.core.windows.net/latest_trilateration")
)

stream_df = stream_df.withWatermark("UpdatedAt", "10 minutes")
stream_df = stream_df.dropDuplicates(["TouristId", "UpdatedAt"])

In [0]:
invalid_coordinates=stream_df.filter(
    (~col("EstimatedLat").between(-90,90))|
    (~col("EstimatedLng").between(-180,180))
)

In [0]:
invalid_accuracy=stream_df.filter(col("Accuracy")<=0)

In [0]:
null_datas=stream_df.filter(
    col("TouristId").isNull()
)

not_gps_data=stream_df.filter(
    ~(col("Source")=="gps")
)

In [0]:
invalid_coordinates.writeStream\
.format("delta")\
    .outputMode("append")\
        .option(
            "checkpointLocation",
            "abfss://checkpoints@travelappprojectstorage.dfs.core.windows.net/quarantine_invalid_coordinates"
        )\
            .trigger(availableNow=True) \
            .start("abfss://quarantine@travelappprojectstorage.dfs.core.windows.net/invalid_coordinates")

In [0]:
invalid_accuracy.writeStream\
.format("delta")\
    .outputMode("append")\
        .option(
            "checkpointLocation",
            "abfss://checkpoints@travelappprojectstorage.dfs.core.windows.net/quarantine_invalid_accuracy"
        )\
            .trigger(availableNow=True) \
            .start("abfss://quarantine@travelappprojectstorage.dfs.core.windows.net/invalid_accuracy")

In [0]:
null_datas.writeStream\
.format("delta")\
    .outputMode("append")\
        .option(
            "checkpointLocation",
            "abfss://checkpoints@travelappprojectstorage.dfs.core.windows.net/quarantine_null_datas"
        )\
            .trigger(availableNow=True) \
            .start("abfss://quarantine@travelappprojectstorage.dfs.core.windows.net/null_datas")

In [0]:
not_gps_data.writeStream\
.format("delta")\
    .outputMode("append")\
        .option(
            "checkpointLocation",
            "abfss://checkpoints@travelappprojectstorage.dfs.core.windows.net/quarantine_not_gps_data"
        )\
            .trigger(availableNow=True) \
            .start("abfss://quarantine@travelappprojectstorage.dfs.core.windows.net/not_gps_data")

In [0]:
clean_df=stream_df.filter(
    (col("EstimatedLat").between(-90,90)) &
    (col("EstimatedLng").between(-180,180)) &
    (col("Accuracy")>0) &
    (col("TouristId").isNotNull()) &
    (col("Source")=="gps")
  )
display(clean_df, checkpointLocation='abfss://checkpoints@travelappprojectstorage.dfs.core.windows.net/')

In [0]:
clean_df=clean_df.withColumn(
    "location_confidence",
    when(col("Accuracy")<=10,"HIGH")\
        .when(col("Accuracy")<=30,"MEDIUM")\
        .otherwise("LOW")
)

clean_df = clean_df.withColumn(
    "inactivity_minutes",
    (current_timestamp().cast("long") - col("UpdatedAt").cast("long")) / 60
)

clean_df=clean_df.withColumn(
    "activity_status",
    when(col("inactivity_minutes")<=15,"ACTIVE")\
        .otherwise("INACTIVE")
)

In [0]:
silver_query=(clean_df.writeStream\
.format("delta")\
    .outputMode("append")\
        .option(
            "checkpointLocation",
            "abfss://checkpoints@travelappprojectstorage.dfs.core.windows.net/location_silver"
        )\
            .trigger(availableNow=True) \
            .start("abfss://silver@travelappprojectstorage.dfs.core.windows.net/tourist_location_silver"))
silver_query.awaitTermination()

In [0]:
silver_df = spark.read.format("delta").load("abfss://silver@travelappprojectstorage.dfs.core.windows.net/tourist_location_silver")
display(silver_df)
print(silver_df.count())
